# Page 5 — "How to Reason": the prompt ladder

Five prompt formulations on one model, thinking off throughout, so the prompt is the only
variable. Produces everything page 5 renders; the page replays committed logs and runs no model.

Writes `out/page_05/`: `traces.jsonl` (every generation), `derived.json` (what the page reads),
`manifest.json` (provenance).

Rung design, item bands and the claims contract live in
[`docs/page_5_refactor.md`](../docs/page_5_refactor.md). Shared plumbing lives in
[`harness.py`](harness.py), which ships alongside this notebook.

## How to run this

| Mode | Model | Scored | Pool | Cap |
|---|---|---|---|---|
| `QUICK` | Qwen3-1.7B | 24 | 12 | 1024 |
| `FULL` | Qwen3-4B | 160 | 24 | 1024 |

`QUICK` resolves figure 2's primary axis. It usually cannot resolve the accuracy panel, which
the notebook detects and flags rather than asserts. `FULL` is what the page ships.

Resumable: `traces.jsonl` is appended as it goes, and `RUN_ID` names the model, band and cap, so
changing any of them starts fresh instead of replaying incomparable traces. Run from this
directory; outputs land in `out/page_05/`.

```bash
pip install "torch>=2.7" "transformers>=4.56" accelerate sympy matplotlib requests
pip install reasoning-from-scratch   # or: pip install git+https://github.com/rasbt/reasoning-from-scratch.git
```

In [ ]:
# ---------------------------------------------------------------------------
# The two knobs. Everything else is derived from these.
# ---------------------------------------------------------------------------
RUN_MODE = "QUICK"        # "QUICK" (Qwen3-1.7B) or "FULL" (Qwen3-4B)
SEED = 123                # selects items only; decoding is greedy and unseeded

PRESETS = {
    "QUICK": dict(model_id="Qwen/Qwen3-1.7B", n_eval=24, n_pool=12, few_shot_k=2,
                  max_new_tokens=1024, eval_levels=(4, 5), exemplar_levels=(1, 2)),
    "FULL": dict(model_id="Qwen/Qwen3-4B", n_eval=160, n_pool=24, few_shot_k=3,
                 max_new_tokens=1024, eval_levels=(4, 5), exemplar_levels=(1, 2)),
}
CFG = PRESETS[RUN_MODE]

# One cap for every rung: the variable is the PROMPT, so the length budget must not
# vary too -- that is page 6's experiment. Asserted per rung below (spec section 3).
MAX_NEW_TOKENS = CFG["max_new_tokens"]

# eval_levels is the lever for accuracy headroom; step down to (3, 4) if the gate
# fails. (2, 3) collides with the exemplar band and needs exemplar_levels=(1,).

print(f"mode={RUN_MODE}  model={CFG['model_id']}  "
      f"items={CFG['n_eval']} (+{CFG['n_pool']} held out)  cap={MAX_NEW_TOKENS}  "
      f"eval L{'+L'.join(map(str, CFG['eval_levels']))}")

In [ ]:
import json
import random
import re
from datetime import datetime, timezone
from pathlib import Path

import torch
from tqdm.auto import tqdm

# Shared plumbing, shipped beside this notebook. What is here and what is
# deliberately not is documented at the top of the file.
import harness

# The inherited grading harness: text-in, text-out, no model dependency.
from reasoning_from_scratch.ch02 import get_device
from reasoning_from_scratch.ch03 import (
    load_math500_test,
    render_prompt,
    extract_final_candidate,
    grade_answer,
)

OUT_DIR = Path("out/page_05")
OUT_DIR.mkdir(parents=True, exist_ok=True)
DATA_DIR = Path("data")
DATA_DIR.mkdir(parents=True, exist_ok=True)

TRACES_PATH = OUT_DIR / "traces.jsonl"
DERIVED_PATH = OUT_DIR / "derived.json"
MANIFEST_PATH = OUT_DIR / "manifest.json"

mean = harness.mean

print("torch", torch.__version__)
print("harness", harness.fingerprint()["version"], harness.fingerprint()["sha256"][:12])

## The grading harness

Inherited unchanged from ch03. `extract_final_candidate` takes the last `\boxed{...}` (falling
back to the last number); `grade_answer` checks equivalence with LaTeX canonicalisation and
SymPy. The next cell asserts it behaves as the sweep assumes, including the chain-then-boxed
case the CoT rungs depend on.

In [ ]:
CHECKS = [
    # (raw completion, ground truth, expected)
    (r"The answer is \boxed{9}.",                          "9",              True),
    (r"Step 1: 23-20=3. Step 2: 3+6=9. So \boxed{9}.",     "9",              True),   # a chain
    (r"27",                                                 "9",              False),  # a wrong blurt
    (r"\boxed{\frac{1}{2}}",                               "0.5",            True),   # LaTeX vs decimal
    (r"\boxed{1,024}",                                     "1024",           True),   # thousands separator
    (r"\boxed{\left( 3, \frac{\pi}{2} \right)}",           r"\left( 3, \frac{\pi}{2} \right)", True),
    (r"I am not sure.",                                     "9",              False),  # no answer at all
]

rows, failures = [], []
for text, truth, expected in CHECKS:
    got = bool(grade_answer(extract_final_candidate(text), truth))
    rows.append((text, truth, expected, got))
    if got != expected:
        failures.append((text, truth, expected, got))

if failures:                                   # the table is for debugging, not for reading
    for text, truth, expected, got in rows:
        flag = "ok " if got == expected else "FAIL"
        print(f"{flag} {extract_final_candidate(text)!r:<40} vs {truth!r:<12} -> {got}")
    raise AssertionError(f"{len(failures)} grader check(s) behaved unexpectedly")

print(f"grader preflight: {len(rows)}/{len(rows)} as expected "
      f"(boxed, chain-then-boxed, wrong blurt, LaTeX vs decimal, separators, tuples, no answer)")


## The item set: two bands, drawn once

This cell defines the experiment: it picks the problems, splits them, and hashes the split so
"the same items" is checkable later.

Scored on `eval_levels` (default 4-5), demonstrated with `exemplar_levels` (default 1-2). The
bands must stay disjoint or a worked answer could appear in the prompt of a problem we then
score — asserted on the levels, not just the item ids. Why these bands, and when to move them:
spec §3.

In [ ]:
math500 = load_math500_test(local_path=DATA_DIR / "math500_test.json")

EVAL_LEVELS = tuple(CFG["eval_levels"])
EXEMPLAR_LEVELS = tuple(CFG["exemplar_levels"])

assert not (set(EVAL_LEVELS) & set(EXEMPLAR_LEVELS)), (
    f"eval band L{EVAL_LEVELS} and exemplar band L{EXEMPLAR_LEVELS} share a level. "
    f"Lowering the eval band to (2, 3) requires exemplar_levels=(1,).")


def draw(levels, n, seed_offset):
    """A deterministic sample from one difficulty band."""
    band = sorted((r for r in math500 if r["level"] in levels),
                  key=lambda r: r["unique_id"])       # sort before shuffling, or the
    random.Random(SEED + seed_offset).shuffle(band)   # draw depends on dict ordering
    return band[:n]


eval_items = draw(EVAL_LEVELS, CFG["n_eval"], 0)
pool_items = draw(EXEMPLAR_LEVELS, CFG["n_pool"], 1)

assert not ({r["unique_id"] for r in pool_items} & {r["unique_id"] for r in eval_items}), \
    "exemplar pool leaked into the eval set"
assert len(eval_items) == CFG["n_eval"], "not enough problems in the eval band"
assert len(pool_items) == CFG["n_pool"], "not enough problems in the exemplar band"

ITEM_SET_HASH = harness.sha256_of(r["unique_id"] for r in eval_items)

band = lambda items: "+".join(f"L{n}x{sum(r['level'] == n for r in items)}"
                              for n in sorted({r["level"] for r in items}))
print(f"scored    {len(eval_items):>3} items  {band(eval_items):<10} sha256 {ITEM_SET_HASH[:12]}")
print(f"exemplars {len(pool_items):>3} items  {band(pool_items):<10} held out, never scored")

## Loading the model

**The cell below is the first one that runs the model.** Everything above it is text, arithmetic
and assertions; from here on there are weights in memory, and the notebook is live.

Four timed stages rather than one silent call — fetch, load, move, warm up — so a long wait is
always attributable to a named thing. The fetch downloads the checkpoint on the first run only
(~1.4 GB for `Qwen3-0.6B`, ~8 GB for `Qwen3-4B`) and is the longest single wait in the notebook;
every run after that reads it from the cache in well under a second.

In [ ]:
device = get_device()
DTYPE = harness.pick_dtype(device)

loaded = harness.load_model(CFG["model_id"], device=device, dtype=DTYPE)
model, tokenizer = loaded["model"], loaded["tokenizer"]
n_params, WARMUP_TOK_S = loaded["params"], loaded["tok_per_s"]

print()
harness.selftest(tokenizer)      # the invariants the traces below depend on

## Proof that thinking is off

The cell to read if you suspect we cheated. Qwen3's chat template takes an `enable_thinking`
flag; switched **off** it hands the model a think block that is *already closed*, so there is
nowhere to put a chain and anything written is the answer.

The two renderings are identical until the end, so only the tail is printed, with newlines shown
as `\n` — the pre-closing is a newline arrangement and would otherwise be invisible.

In [ ]:
demo = [{"role": "user", "content": "What is 2+2?"}]
render = lambda flag: tokenizer.apply_chat_template(
    demo, tokenize=False, add_generation_prompt=True, enable_thinking=flag)

off, on = render(False), render(True)
HAS_THINKING_TOGGLE = off != on

d = 0                                   # print only from where the two diverge
while d < min(len(off), len(on)) and off[d] == on[d]:
    d += 1

show = lambda s: s[d:].replace("\n", "\\n")     # newlines are the point here
print(f"both templates agree for {d} characters, ending: ...{off[max(0, d - 30):d]!r}")
print()
print(f"  OFF (used here)  ...{show(off)}")
print(f"  ON  (never used) ...{show(on)}")

if HAS_THINKING_TOGGLE:
    # By structure, not string inequality: the OFF template must end on a think
    # block that opens and immediately closes.
    assert re.sub(r"\s+", "", off).endswith("<think></think>"), (
        "thinking-off did not pre-close the think block, so this notebook's one "
        "constraint is not actually enforced by the template")
    assert "<think>" not in on, "thinking-on unexpectedly pre-fills a think block"
    print("\nOK   the OFF template ends on an already-closed think block: the model has")
    print("     nowhere to reason, so whatever it writes next is its answer.")
else:
    print("\nNOTE this checkpoint has no thinking toggle. The constraint still holds --")
    print("     there is no thinking mode to leave on -- but the manifest records that")
    print("     we did not disable anything, rather than implying we did.")

EOS_IDS = harness.end_of_turn_ids(tokenizer)
print(f"\nend-of-turn ids: {sorted(EOS_IDS)}")

## The generation loop

`harness.generate` does the mechanics. This cell fixes the one thing that is a claim rather than
plumbing — `enable_thinking=False` — and smoke-tests the result.

Token strings are decoded so that joining them reproduces the text exactly (asserted in the
harness, because the page reveals a trace one token at a time). Truncation is recorded: on this
page a binding cap is a defect, not a variable. End-of-turn tokens are stripped — they are how
the model signals it stopped, not something it wrote.

In [ ]:
def generate(prompt_text, max_new_tokens=None):
    """Thinking is OFF for every generation on this page: the prompt is the only
    variable. The harness gives enable_thinking no default so this stays visible."""
    return harness.generate(prompt_text, model, tokenizer,
                            enable_thinking=False,
                            max_new_tokens=max_new_tokens or MAX_NEW_TOKENS,
                            eos_ids=EOS_IDS)


# Tells a broken harness apart from a weak model: two probes, the ids beside what they
# decode to, and why generation stopped. Both wrong means the harness is suspect; only
# the harder one wrong is a small model being small, which is information.
for question, expect in [("What is 2 + 2?", "4"), ("What is 10 - 3?", "7")]:
    g = generate(question, max_new_tokens=32)
    stop = "end-of-turn" if g["stopped_on_eos"] else ("hit the cap" if g["truncated"] else "?")
    verdict = "as expected" if expect in g["text"] else f"NOT {expect!r}"
    print(f"{question!r:<20} -> {g['text']!r:<24} {g['tokens_generated']:>2} tokens, "
          f"stopped: {stop:<12} {verdict}")

print("\n--- the last probe in full " + "-" * 46)
print(tokenizer.apply_chat_template([{"role": "user", "content": "What is 10 - 3?"}],
                                    tokenize=False, add_generation_prompt=True,
                                    enable_thinking=False).replace("\n", "\\n"))
ids = tokenizer(g["text"], add_special_tokens=False).input_ids
print("\ntokens:", [(i, s) for i, s in zip(ids, g["tokens"])][:12])

## The ladder

Five formulations, least to most instruction. Design rationale: spec §§2-3.

| Rung | What is added | Whose mechanism |
|---|---|---|
| `bare` | nothing — the harness prompt alone | the true default (ch03's baseline) |
| `forced_blurt` | "Give only the final answer. Do not explain." | the floor |
| `zero_shot_cot` | `"\n\nExplain step by step."` | Kojima et al. 2022 · ch04 §4.3 |
| `one_shot` | one worked example prepended | Wei et al. 2022 |
| `few_shot` | k worked examples prepended | Wei et al. 2022 |

Every rung carries ch03's `render_prompt`, which fixes the `\boxed{}` output format — vary that
too and a rung could score lower only because the grader cannot find its answer. The exemplar
rungs get no "explain step by step": combining Wei's and Kojima's mechanisms would measure a
third thing.

In [ ]:
RUNGS = [
    ("bare",          "bare question"),
    ("forced_blurt",  "\"answer with the number\""),
    ("zero_shot_cot", "\"explain step by step\""),
    ("one_shot",      "one worked example"),
    ("few_shot",      f"{CFG['few_shot_k']} worked examples"),
]
RUNG_KEYS = [k for k, _ in RUNGS]
RUNG_LABEL = dict(RUNGS)

FORCED_BLURT_SUFFIX = "\n\nGive only the final answer. Do not explain."
COT_SUFFIX = "\n\nExplain step by step."      # ch04 section 4.3, verbatim mechanism


def build_prompt(rung, question, exemplars=None):
    '''The whole ladder, in one readable function. Nothing else varies.'''
    base = render_prompt(question)             # ch03's template -- CONSTANT across rungs

    if rung == "bare":
        return base
    if rung == "forced_blurt":
        return base + FORCED_BLURT_SUFFIX
    if rung == "zero_shot_cot":
        return base + COT_SUFFIX

    k = 1 if rung == "one_shot" else CFG["few_shot_k"]
    shots = "".join(
        render_prompt(e["question"]) + "\n" + e["trace"].strip() + "\n\n"
        for e in (exemplars or [])[:k]
    )
    return shots + base


# The prompts ARE the experiment on this page: the five rungs differ in nothing else.
# So print the shared stem once, then only what each rung adds and where. Exemplars go
# BEFORE the stem, instructions go after it -- the same split figure 1 uses to highlight
# the diff, so the notebook and the page describe a prompt the same way.
STEM = render_prompt("<QUESTION>")

print("the stem, identical in all five rungs")
print("-" * 74)
print(STEM)
print("-" * 74)

for key, label in RUNGS:
    p = build_prompt(key, "<QUESTION>", exemplars=None)
    before, after = p[:p.index(STEM)], p[p.index(STEM) + len(STEM):]
    if key in ("one_shot", "few_shot"):
        k = 1 if key == "one_shot" else CFG["few_shot_k"]
        note = f"prepends {k} worked example{'s' if k > 1 else ''} (harvested later, printed in full there)"
    elif after:
        note = f"appends {after.strip()!r}"
    else:
        note = "adds nothing"
    print(f"  {key:<14} {note}")

print(f"\nthe {len(COT_SUFFIX.split())} words figure 1 highlights: {COT_SUFFIX.strip()!r}")


## The sweep, part 1: the machinery

One function: hand it a rung and a list of problems, it asks the model each question and returns
the records. `harness.TraceStore` writes each answer the moment it arrives and keys them by
`(rung, problem)`, so a stopped run loses nothing and a finished rung costs nothing to re-run.

The record is the schema shared with pages 6 to 9, with `rung` where page 6 puts `budget`.
Provenance is not repeated per line; it lives once in `manifest.json`.

In [ ]:
# The key names everything that changes what a generation IS. Leaving the cap out of
# it once meant that raising MAX_NEW_TOKENS and re-running silently replayed traces
# produced under the old cap, and reported the old truncation rate.
RUN_ID = (f"{RUN_MODE.lower()}-{CFG['model_id'].split('/')[-1]}"
          f"-L{''.join(str(n) for n in EVAL_LEVELS)}"
          f"-cap{MAX_NEW_TOKENS}-{ITEM_SET_HASH[:8]}")

STORE = harness.TraceStore(TRACES_PATH, RUN_ID, ("rung", "problem_id"))
STORE.report()


def run_rung(rung, items, exemplars=None, label=None, split="eval"):
    records, n_new = [], 0
    exemplar_ids = [e["problem_id"] for e in (exemplars or [])][
        : (1 if rung == "one_shot" else CFG["few_shot_k"])
    ] if rung in ("one_shot", "few_shot") else []

    bar = tqdm(items, desc=f"{label or rung:<20}", unit="item")
    for row in bar:
        rec = STORE.get(rung, row["unique_id"])
        if rec is None:
            prompt = build_prompt(rung, row["problem"], exemplars)
            g = generate(prompt)
            extracted = extract_final_candidate(g["text"])
            rec = STORE.append({
                "problem_id": row["unique_id"],
                "split": split,                # "eval" is scored; "held_out" feeds exemplars
                "level": row["level"],
                "subject": row["subject"],
                "question": row["problem"],
                "ground_truth": row["answer"],
                "rung": rung,
                "prompt": prompt,
                "trace": g["text"],
                "tokens": g["tokens"],
                "truncated": g["truncated"],
                "max_new_tokens": MAX_NEW_TOKENS,
                "extracted_answer": extracted,
                "correct": bool(grade_answer(extracted, row["answer"])),
                "tokens_generated": g["tokens_generated"],
                "exemplar_ids": exemplar_ids,
                "thinking_leaked": g["thinking_leaked"],
            })
            n_new += 1
        records.append(rec)
        bar.set_postfix(acc=f"{sum(r['correct'] for r in records)/len(records):.2f}", new=n_new)

    return records

## The sweep, part 2: run it

Three rungs need no worked examples, so they go first. The fourth pass runs the same zero-shot
prompt over the held-out easy problems: those answers are not graded against anything, they are
where the worked examples come from, which is why they must exist before the exemplar rungs can
run at all.

One number at the end decides the page's framing verb — whether the model already shows working
when nobody asked. Recorded in `derived.json` so it cannot be reversed in prose later (spec §3).

In [ ]:
# Worst-case projection from the warm-up: assumes every generation runs to the cap.
_gens = len(RUNG_KEYS) * len(eval_items) + len(pool_items)
print(f"about {_gens} generations; at ~{WARMUP_TOK_S:.0f} tok/s and a {MAX_NEW_TOKENS}-token cap "
      f"that is under {_gens * MAX_NEW_TOKENS / max(WARMUP_TOK_S, 1e-6) / 3600:.1f} h worst case.\n")

# Phase A: the three rungs that need no exemplars. zero_shot_cot also runs on the
# held-out pool -- that pass is what the worked examples are harvested from.
results = {}
for rung in ("bare", "forced_blurt", "zero_shot_cot"):
    results[rung] = run_rung(rung, eval_items, label=rung)

pool_cot = run_rung("zero_shot_cot", pool_items, label="pool/zero_shot_cot", split="held_out")


# The framing verdict (spec section 3): does the bare rung already show working?
# Decides figure 1's left rung and the page's verb; recorded in derived.json.
def shows_working(rec):
    lines = [ln for ln in rec["trace"].strip().splitlines() if ln.strip()]
    return len(lines) > 2 or rec["tokens_generated"] > 60


bare_working_share = mean([shows_working(r) for r in results["bare"]])
BARE_ALREADY_REASONS = bare_working_share >= 0.5
FRAMING_VERB = "let it reason" if BARE_ALREADY_REASONS else "make it reason"
FIGURE1_LEFT = "forced_blurt" if BARE_ALREADY_REASONS else "bare"

print(f"\nbare rung shows working on {bare_working_share:.0%} of items")
print(f'-> page verb: "{FRAMING_VERB}"   figure 1 left: {FIGURE1_LEFT}')

## Harvesting the worked examples

**The exemplars are the model's own verified-correct solutions to held-out problems** —
authored ones would be the only thing on the page a reader could not re-derive (spec §3).
Selection is deterministic: correct under the grader, untruncated, short enough, shortest first,
ties broken by `problem_id`.

In [ ]:
MAX_EXEMPLAR_TOKENS = 400

# Report the pool's yield whether or not it worked, so a near miss is visible on the
# run before it becomes a failure on the next one.
n_correct = sum(r["correct"] for r in pool_cot)
print(f"held-out pool: {n_correct}/{len(pool_cot)} solved correctly "
      f"({n_correct / max(len(pool_cot), 1):.0%})")

candidates = [
    r for r in pool_cot
    if r["correct"] and not r["truncated"] and r["tokens_generated"] <= MAX_EXEMPLAR_TOKENS
]
candidates.sort(key=lambda r: (r["tokens_generated"], r["problem_id"]))

if not candidates:
    raise RuntimeError(
        f"No usable exemplars. Of {len(pool_cot)} held-out problems, {n_correct} were solved "
        f"correctly and none of those were both untruncated and <= {MAX_EXEMPLAR_TOKENS} "
        f"tokens.\n\n"
        f"The two levers, in order of preference:\n"
        f"  1. raise CFG['n_pool'] (currently {CFG['n_pool']}) -- exemplars come from levels "
        f"{EXEMPLAR_LEVELS}, so these are cheap generations;\n"
        f"  2. use a larger model.\n\n"
        f"Do not author exemplars by hand. The page's claim is that they are the model's own "
        f"verified work, and that claim is the reason this raises instead of falling back."
    )

exemplars = [
    {
        "problem_id": r["problem_id"],
        "level": r["level"],
        "question": r["question"],
        "trace": r["trace"],
        "ground_truth": r["ground_truth"],
        "extracted_answer": r["extracted_answer"],
        "tokens_generated": r["tokens_generated"],
    }
    for r in candidates[: CFG["few_shot_k"]]
]

if len(exemplars) < CFG["few_shot_k"]:
    print(f"WARNING: only {len(exemplars)} usable exemplars from a pool of {len(pool_items)}.\n"
          f"         Raise n_pool, or MAX_EXEMPLAR_TOKENS, and re-run. The few-shot rung\n"
          f"         below will use however many were found, which weakens the comparison.")

print(f"{len(candidates)} of {len(pool_items)} held-out problems were solved correctly, "
      f"untruncated, and short enough.\n")
for e in exemplars:
    print(f"  {e['problem_id']:<34} L{e['level']}  {e['tokens_generated']:>4} tokens  "
          f"-> {e['extracted_answer']}")

print("\n--- the first exemplar, exactly as it will be prepended " + "-" * 20)
print(render_prompt(exemplars[0]["question"]))
print(exemplars[0]["trace"].strip()[:800])

In [ ]:
# Phase B: the exemplar rungs. Their prompts contain no instruction to reason --
# only examples of reasoning.
for rung in ("one_shot", "few_shot"):
    results[rung] = run_rung(rung, eval_items, exemplars=exemplars, label=rung)

## Sweep integrity

Three assertions the page's honesty rests on; each raises rather than warns.

Truncation is checked **per rung**. An earlier run passed a 13% aggregate as a warning while the
cap clipped 0% of the answer-only rung and 33% of the step-by-step one — and a truncated trace
loses its `\boxed{}` and grades wrong, so that was a scoring penalty aimed at the rungs this page
argues for.

In [ ]:
all_records = [r for rung in RUNG_KEYS for r in results[rung]]

leaked = [r for r in all_records if r["thinking_leaked"]]
print(f"1. thinking never leaked into a trace ......... {len(leaked)} violations")
assert not leaked, "a trace contained a <think> block; the page's core constraint is broken"

# PER RUNG, not in aggregate: an overall rate looks tolerable while the cap clips
# only the rungs that write chains. Length is page 6's variable; here it binds on
# nobody.
TRUNCATION_BUDGET = 0.05

trunc_by_rung = {k: mean([r["truncated"] for r in results[k]]) for k in RUNG_KEYS}
trunc_rate = mean([r["truncated"] for r in all_records])
worst_rung = max(trunc_by_rung, key=trunc_by_rung.get)

print(f"2. the generation cap never bound ............. {trunc_rate:.1%} overall, "
      f"worst rung {worst_rung} at {trunc_by_rung[worst_rung]:.1%}")
for k in RUNG_KEYS:
    flag = "  <-- over budget" if trunc_by_rung[k] > TRUNCATION_BUDGET else ""
    print(f"     {k:<16}{trunc_by_rung[k]:>6.1%}{flag}")

assert trunc_by_rung[worst_rung] <= TRUNCATION_BUDGET, (
    f"the {worst_rung} rung was truncated on {trunc_by_rung[worst_rung]:.1%} of items, over the "
    f"{TRUNCATION_BUDGET:.0%} budget. Raise MAX_NEW_TOKENS (currently {MAX_NEW_TOKENS}) and "
    f"re-run -- the cap is in RUN_ID, so raising it starts a fresh set of generations rather "
    f"than replaying these. Page 5 lets every prompt run to natural length; capping it is "
    f"page 6's experiment, not this one.")

ex_ids = {e["problem_id"] for e in exemplars}
overlap = ex_ids & {r["problem_id"] for r in results["few_shot"]}
print(f"3. exemplars stayed out of the eval set ....... {len(overlap)} overlaps")
assert not overlap, "an exemplar problem was also scored"

print(f"\n{len(all_records)} generations across {len(RUNG_KEYS)} rungs and "
      f"{len(eval_items)} items.")

## Reasoning steps, and where they sit in the token stream

Figure 1 walks a chain a step at a time and figure 2 counts steps, so the division is a
derivation and belongs here rather than in TypeScript.

The rule, in order: blank-line paragraphs, then newlines, then sentences — first that divides
the trace. Fragments under `MIN_STEP_CHARS` merge backwards so no orphan line appears. Steps
carry contiguous token ranges covering every token, so the page's reveal stops at real
boundaries.

In [ ]:
# A "fragment" is an orphan like "OK" or "So:", not a short-but-real step. Length
# alone cannot tell them apart: "5+2=7" is five characters and carries a whole
# claim. A digit is the signal, which is the same reason page 9 argues a step has
# to be big enough to carry a fact. Two kinds of span are never merged away.
MIN_STEP_CHARS = 12


def _raw_spans(text):
    """First splitting rule that actually divides the trace: paragraphs, lines, sentences."""
    for pattern in (r"\n\s*\n", r"\n", r"(?<=[.!?])\s+"):
        spans, pos = [], 0
        for m in re.finditer(pattern, text):
            if m.start() > pos:
                spans.append((pos, m.start()))
            pos = m.end()
        if pos < len(text):
            spans.append((pos, len(text)))
        spans = [(a, b) for a, b in spans if text[a:b].strip()]
        if len(spans) > 1:
            return spans
    return [(0, len(text))] if text.strip() else []


def segment_steps(text, tokens):
    """Steps with token ranges. Relies on "".join(tokens) == text, asserted at generation."""
    assert "".join(tokens) == text, "token strings do not reconstruct the trace"

    raw = _raw_spans(text)
    spans = []
    for k, (a, b) in enumerate(raw):
        piece = text[a:b].strip()
        is_last = k == len(raw) - 1
        mergeable = (
            len(piece) < MIN_STEP_CHARS
            and not re.search(r"\d", piece)         # no number, so it states nothing
            and not is_last                        # the landing line stays its own step
            and "\\boxed" not in piece             # so does whatever carries the answer
        )
        if spans and mergeable:
            spans[-1] = (spans[-1][0], b)          # merge fragments backwards
        else:
            spans.append((a, b))

    ends, run = [], 0                              # cumulative char length per token
    for t in tokens:
        run += len(t)
        ends.append(run)

    def token_at(char_index):
        lo, hi = 0, len(ends)
        while lo < hi:
            mid = (lo + hi) // 2
            if ends[mid] <= char_index:
                lo = mid + 1
            else:
                hi = mid
        return lo

    steps = [{"text": text[a:b].strip(), "token_start": token_at(a)} for a, b in spans]
    for j, st in enumerate(steps):                 # contiguous ranges covering every token
        st["token_end"] = steps[j + 1]["token_start"] if j + 1 < len(steps) else len(tokens)
    return steps


# Checked against a real trace from the sweep, not a toy string.
_probe = next((r for r in results["zero_shot_cot"] if r["correct"]), results["zero_shot_cot"][0])
_steps = segment_steps(_probe["trace"], _probe["tokens"])
_norm = lambda s: re.sub(r"\s+", "", s)
assert _norm("".join(s["text"] for s in _steps)) == _norm(_probe["trace"]), \
    "segmentation dropped text"
assert _steps[-1]["token_end"] == len(_probe["tokens"]), "segmentation did not cover every token"

print(f"{_probe['problem_id']}: {_probe['tokens_generated']} tokens -> {len(_steps)} steps")
for s in _steps[:6]:
    print(f"  [{s['token_start']:>3}:{s['token_end']:>3}]  {s['text'][:88]}")


## Figure 1 — three panels on one problem

Three real completions on one problem, same weights, same decoding.

| Panel | Rung | What it shows |
|---|---|---|
| left | framing verdict's rung (`bare` or `forced_blurt`) | the default: no chain |
| middle | `zero_shot_cot` | four words, and a chain appears |
| right | `one_shot` | a worked example, and the chain takes its shape |

Curated from the sweep, not generated for the figure: left must miss and middle must land. The
right panel is a preference in the sort, never a filter, so it costs the search nothing. Its
caption is about the **shape** of the chain, not who was right (spec §6).

In [ ]:
MAX_SPECIMEN_TOKENS = 260      # a chain the reader can walk on screen

left_by_id = {r["problem_id"]: r for r in results[FIGURE1_LEFT]}
one_by_id = {r["problem_id"]: r for r in results["one_shot"]}

triples = []
for mid in results["zero_shot_cot"]:
    left = left_by_id.get(mid["problem_id"])
    if left and not left["correct"] and mid["correct"] and not mid["truncated"]:
        triples.append((left, mid, one_by_id.get(mid["problem_id"])))

# Preferences, not filters, so the third panel costs the search nothing.
def _rank(t):
    _, mid, right = t
    return (
        not (right and right["correct"] and not right["truncated"]),
        mid["tokens_generated"] > MAX_SPECIMEN_TOKENS,
        mid["tokens_generated"],
        mid["problem_id"],
    )


triples.sort(key=_rank)
SPECIMEN_IS_FALLBACK = False

if triples:
    left_spec, mid_spec, right_spec = triples[0]
else:
    # Degrade and flag rather than fail: figure 2 is unaffected, and a flagged
    # specimen must not be translated (spec section 6).
    SPECIMEN_IS_FALLBACK = True
    mid_spec = min((r for r in results["zero_shot_cot"] if not r["truncated"]),
                   key=lambda r: (not r["correct"], r["tokens_generated"], r["problem_id"]))
    left_spec = left_by_id[mid_spec["problem_id"]]
    right_spec = one_by_id.get(mid_spec["problem_id"])
    print("WARNING: no problem where the left rung misses and zero-shot CoT lands.\n"
          "         Falling back to an illustrative triple; derived.json flags it.\n"
          "         Raise n_eval or run FULL before translating this to the page.\n")

# Splitting on the shared stem gives the page one highlight rule for all three
# panels instead of a per-rung special case.
PROMPT_STEM = render_prompt(mid_spec["question"])

PANELS = []
for name, rec in (("left", left_spec), ("middle", mid_spec), ("right", right_spec)):
    if rec is None:
        continue
    p = rec["prompt"]
    assert PROMPT_STEM in p, f"{name} panel's prompt does not contain the shared stem"
    cut = p.index(PROMPT_STEM)
    PANELS.append({
        "panel": name,
        "rung": rec["rung"],
        "prompt_before": p[:cut],                       # exemplars, on the one-shot panel
        "prompt_after": p[cut + len(PROMPT_STEM):],     # suffix, on the instruction rungs
        "trace": rec["trace"],
        "tokens": rec["tokens"],
        "steps": segment_steps(rec["trace"], rec["tokens"]),
        "tokens_generated": rec["tokens_generated"],
        "extracted_answer": rec["extracted_answer"],
        "correct": rec["correct"],
    })

print(f"{len(triples)} candidate triples"
      + ("   [FALLBACK SPECIMEN]" if SPECIMEN_IS_FALLBACK else "")
      + f";  chosen {mid_spec['problem_id']} (L{mid_spec['level']})\n")
print("QUESTION:", mid_spec["question"][:300])
print("TRUTH   :", mid_spec["ground_truth"])

for pan in PANELS:
    mark = "correct" if pan["correct"] else "wrong  "
    added = (pan["prompt_after"] or pan["prompt_before"]).strip()
    print(f"\n--- {pan['panel']:<6} {pan['rung']:<14} {mark}  "
          f"{pan['tokens_generated']:>4} tokens, {len(pan['steps'])} steps")
    print(f"    prompt adds: {added[:100]!r}" if added else "    prompt adds: nothing")
    for s in pan["steps"][:4]:
        print("     ", s["text"][:92])
    if len(pan["steps"]) > 4:
        print(f"      ... {len(pan['steps']) - 4} more steps")


## Figure 2 — the ladder, measured

Two panels, one row per rung; two measures means two axes, never one chart with two scales.
Model size appears on neither (spec §1).

- **Top, primary:** share of responses containing a chain — `MIN_CHAIN_STEPS` steps under the
  same rule figure 1 animates. This is the page's title as an axis.
- **Bottom, secondary:** accuracy, gated. Both panels carry 95% Wilson intervals; if the best
  and worst rungs overlap, the panel greys out, says so, and `derived.json` forbids ranking it.

Why the primary axis is not accuracy: spec §2. The cost row beneath is the handoff to page 6.

In [ ]:
# A chain is two steps or more, under the SAME rule figure 1 uses to animate a
# trace. One rule, used twice: disagreeing with where a step ends means disagreeing
# with the figure and the measurement together. A one-step response is an answer.
MIN_CHAIN_STEPS = 2


def summarise(rung):
    rs = results[rung]
    toks = sorted(r["tokens_generated"] for r in rs)
    steps = sorted(len(segment_steps(r["trace"], r["tokens"])) for r in rs)
    n_chain = sum(s >= MIN_CHAIN_STEPS for s in steps)
    n_correct = sum(r["correct"] for r in rs)
    return {
        "rung": rung, "label": RUNG_LABEL[rung], "n": len(rs),
        # primary axis: did the prompt produce reasoning?
        "n_chain": n_chain,
        "chain_share": n_chain / len(rs),
        "chain_ci": harness.wilson(n_chain, len(rs)),
        "median_steps": steps[len(steps) // 2],
        # secondary: did it arrive at the answer?
        "n_correct": n_correct,
        "accuracy": n_correct / len(rs),
        "accuracy_ci": harness.wilson(n_correct, len(rs)),
        # the cost row, which is the handoff to page 6
        "mean_tokens": mean(toks),
        "median_tokens": toks[len(toks) // 2],
        "total_tokens": sum(toks),
        "truncated": sum(r["truncated"] for r in rs),
    }


summary = [summarise(k) for k in RUNG_KEYS]
chain = {s["rung"]: s["chain_share"] for s in summary}

print(f"{'rung':<16}{'prompt':<26}{'chain':>12}{'steps':>7}{'solved':>10}{'mean tok':>10}")
print("-" * 81)
for s in summary:
    print(f"{s['rung']:<16}{s['label']:<26}{s['chain_share']:>11.0%}{s['median_steps']:>7}"
          f"{s['n_correct']:>5}/{s['n']:<4}{s['mean_tokens']:>10.0f}")


def separated(key, ci_key):
    """Do the extreme rungs' intervals clear each other on this axis?"""
    hi = max(summary, key=lambda s: s[key])
    lo = min(summary, key=lambda s: s[key])
    return hi[ci_key][0] > lo[ci_key][1], hi, lo


CHAIN_RESOLVABLE, chain_hi, chain_lo = separated("chain_share", "chain_ci")
ACCURACY_RESOLVABLE, best, floor = separated("accuracy", "accuracy_ci")

print(f"\nchain axis     {chain_lo['rung']} {chain_lo['chain_share']:.0%} -> "
      f"{chain_hi['rung']} {chain_hi['chain_share']:.0%}   "
      f"{'RESOLVED' if CHAIN_RESOLVABLE else 'NOT resolvable at this n'}")
print(f"accuracy axis  {floor['rung']} {floor['accuracy']:.0%} -> "
      f"{best['rung']} {best['accuracy']:.0%}   "
      f"{'RESOLVED' if ACCURACY_RESOLVABLE else 'NOT resolvable at this n'}")

if not CHAIN_RESOLVABLE:
    print("\n!! The primary axis did not separate -- the page's core claim. This run is")
    print("   not translatable. Check the prompts before blaming the sample size.")
if not ACCURACY_RESOLVABLE:
    print("\n   The accuracy panel is drawn but must not be ranked in prose or captions.")
    print(f"   For headroom, set eval_levels=(3, 4) in PRESETS "
          f"(currently L{'+L'.join(map(str, EVAL_LEVELS))}).")

ordered = [s["rung"] for s in sorted(summary, key=lambda s: s["chain_share"])]
expected = ["forced_blurt", "bare", "zero_shot_cot", "one_shot", "few_shot"]
collapse = abs(chain["few_shot"] - chain["one_shot"]) * 100 < 2.0

print(f"\nranking by chain share, worst to best: {' < '.join(ordered)}"
      + ("" if ordered == expected
         else "  (differs from the naive expectation -- the caption should say so)"))
if collapse:
    print("one_shot and few_shot within 2 points -> consider collapsing them into one rung")

In [ ]:
import matplotlib.pyplot as plt

# Two panels, never two scales on one axes. Bars carry 95% Wilson intervals, so a
# reader sees the accuracy panel fail to separate rather than being told.
INK = "#3b4a6b"          # the primary series, as elsewhere in these notebooks
MUTED = "#9aa0ab"        # an unresolved measure: greyed AND labelled, never colour alone

fig, (ax_chain, ax_acc) = plt.subplots(
    2, 1, figsize=(9, 6.2), sharey=True, gridspec_kw={"hspace": 0.42})
ys = list(range(len(summary)))[::-1]


def panel(ax, key, ci_key, colour, xlabel, title):
    vals = [s[key] * 100 for s in summary]
    lo = [max(0.0, s[key] * 100 - s[ci_key][0] * 100) for s in summary]
    hi = [max(0.0, s[ci_key][1] * 100 - s[key] * 100) for s in summary]
    ax.barh(ys, vals, height=0.55, color=colour)
    ax.errorbar(vals, ys, xerr=[lo, hi], fmt="none",
                ecolor="#41474f", elinewidth=1.1, capsize=3)
    ax.set_yticks(ys)
    ax.set_yticklabels([s["label"] for s in summary])
    ax.set_xlabel(xlabel)
    ax.set_xlim(0, 100)
    ax.spines[["top", "right"]].set_visible(False)
    ax.set_title(title, fontsize=10, loc="left")
    for y, s in zip(ys, summary):                 # direct labels; five rows, so all of them
        ax.text(s[ci_key][1] * 100 + 1.6, y, f"{s[key]*100:.0f}%",
                va="center", fontsize=9, color="#41474f")


panel(ax_chain, "chain_share", "chain_ci", INK,
      "responses containing a chain (%)",
      f"What the prompt produces  ·  a chain is ≥{MIN_CHAIN_STEPS} steps")

panel(ax_acc, "accuracy", "accuracy_ci", INK if ACCURACY_RESOLVABLE else MUTED,
      "share of problems solved (%)",
      "Whether it arrives at the answer"
      + ("" if ACCURACY_RESOLVABLE else "  —  NOT RESOLVABLE, do not rank"))

steps = "  ·  ".join(str(s["median_steps"]) for s in summary)
cost = "  ·  ".join(f"{s['mean_tokens']:.0f}" for s in summary)
fig.suptitle(f"{CFG['model_id']} · thinking off · greedy · "
             f"MATH-500 L{'+L'.join(map(str, EVAL_LEVELS))}, n={len(eval_items)}",
             fontsize=10, x=0.01, ha="left", y=0.995)
fig.text(0.01, -0.02, f"median steps, same order:   {steps}", fontsize=9, color="#555")
fig.text(0.01, -0.055, f"tokens written, same order:  {cost}", fontsize=9, color="#555")
if not ACCURACY_RESOLVABLE:
    fig.text(0.01, -0.095,
             f"Intervals are 95% Wilson. The accuracy panel's best and worst rungs overlap at "
             f"n={len(eval_items)}, so it carries no ranking.",
             fontsize=8.5, color="#8a5a2b")

fig.tight_layout()
fig.savefig(OUT_DIR / "figure_2.png", dpi=160, bbox_inches="tight")
plt.show()

## The close

The rung that most reliably produces reasoning, beside what it cost. Accuracy joins only if its
gate passed; the fields are `None` otherwise, so nothing downstream can borrow an unearned
number.

In [ ]:
# The handoff: the formulation that most reliably produces reasoning also writes
# the most. Accuracy fields are None unless the gate passed, so a translation step
# cannot reach for a number this run did not earn.
close = {
    "chain_best_rung": chain_hi["rung"],
    "chain_best_label": chain_hi["label"],
    "chain_best_share": chain_hi["chain_share"],
    "chain_best_median_steps": chain_hi["median_steps"],
    "chain_best_mean_tokens": chain_hi["mean_tokens"],
    "chain_floor_rung": chain_lo["rung"],
    "chain_floor_label": chain_lo["label"],
    "chain_floor_share": chain_lo["chain_share"],
    "chain_floor_median_steps": chain_lo["median_steps"],
    "chain_floor_mean_tokens": chain_lo["mean_tokens"],
    "chain_gain_pts": (chain_hi["chain_share"] - chain_lo["chain_share"]) * 100,
    "token_multiple": chain_hi["mean_tokens"] / max(chain_lo["mean_tokens"], 1e-9),
    "chain_resolvable": CHAIN_RESOLVABLE,
    # secondary, and only when earned
    "accuracy_resolvable": ACCURACY_RESOLVABLE,
    "accuracy_best_rung": best["rung"] if ACCURACY_RESOLVABLE else None,
    "accuracy_best": best["accuracy"] if ACCURACY_RESOLVABLE else None,
    "accuracy_floor_rung": floor["rung"] if ACCURACY_RESOLVABLE else None,
    "accuracy_floor": floor["accuracy"] if ACCURACY_RESOLVABLE else None,
    "accuracy_gain_pts": ((best["accuracy"] - floor["accuracy"]) * 100
                          if ACCURACY_RESOLVABLE else None),
}

print(f"{close['chain_best_label']:<26} {close['chain_best_share']:>6.0%} produce a chain   "
      f"{close['chain_best_median_steps']:>2} steps   "
      f"{close['chain_best_mean_tokens']:>6.0f} tokens/problem")
print(f"{close['chain_floor_label']:<26} {close['chain_floor_share']:>6.0%} produce a chain   "
      f"{close['chain_floor_median_steps']:>2} steps   "
      f"{close['chain_floor_mean_tokens']:>6.0f} tokens/problem")
print(f"\n{close['chain_gain_pts']:+.0f} points of reasoning-produced, "
      f"for {close['token_multiple']:.0f}x the tokens.  <- the sentence page 6 opens on")

if ACCURACY_RESOLVABLE:
    print(f"\nand it was also more often right: {close['accuracy_gain_pts']:+.1f} points, "
          f"{close['accuracy_floor_rung']} -> {close['accuracy_best_rung']}")
else:
    print("\naccuracy did not separate at this n, so the page claims nothing about it.")

## Artifacts

`derived.json` is the contract with the page: the page reads it and nothing else, and **no
inference runs at demonstration time.** Its top-level `claims` object is the machine-readable
form of spec §6 and is read before any number. `traces.jsonl` stays out of the web bundle and
ships as a download.

In [ ]:
manifest = {
    "run_id": RUN_ID,
    "notebook": "page_05_how_to_reason.ipynb",
    "notebook_sha256": harness.file_sha256("page_05_how_to_reason.ipynb"),
    "harness": harness.fingerprint(),
    "page": 5,
    "run_mode": RUN_MODE,
    "timestamp_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    "notebook_git_sha": harness.git_sha(),
    "model": {
        "id": CFG["model_id"],
        "revision": harness.repo_revision(CFG["model_id"]),
        "tokenizer_sha256": harness.tokenizer_fingerprint(tokenizer),
        "params": n_params,
        "dtype": str(DTYPE),
        "thinking": "off",
        "has_thinking_toggle": HAS_THINKING_TOGGLE,
    },
    "decoding": {"strategy": "greedy", "do_sample": False,
                 "max_new_tokens": MAX_NEW_TOKENS},
    "items": {
        "source": "MATH-500 (ch03 copy)",
        "eval_levels": list(EVAL_LEVELS),
        "exemplar_levels": list(EXEMPLAR_LEVELS),
        "n_eval": len(eval_items),
        "n_held_out": len(pool_items),
        "eval_set_sha256": ITEM_SET_HASH,
        "seed": SEED,
    },
    "grading": {"extract": "reasoning_from_scratch.ch03.extract_final_candidate",
                "grade": "reasoning_from_scratch.ch03.grade_answer"},
    "chat_template_thinking_off": off,
    "versions": {"torch": torch.__version__,
                 "transformers": __import__("transformers").__version__,
                 "device": str(device)},
    "integrity": {"thinking_leaks": len(leaked),
                  "truncation_rate": trunc_rate,
                  "truncation_by_rung": trunc_by_rung,
                  "truncation_budget": TRUNCATION_BUDGET},
}

# Straight from the sweep records, so the page's "see full prompt" disclosure renders
# what was actually sent instead of re-assembling it in TypeScript.
few_shot_prompt = next(
    r["prompt"] for r in results["few_shot"] if r["problem_id"] == mid_spec["problem_id"])

derived = {
    "manifest": manifest,
    # Read before any number: the machine-readable form of spec section 6.
    "claims": {
        "chain_axis_resolvable": CHAIN_RESOLVABLE,
        "accuracy_axis_resolvable": ACCURACY_RESOLVABLE,
        "specimen_usable": not SPECIMEN_IS_FALLBACK,
        "may_claim_prompt_changes_what_is_written": CHAIN_RESOLVABLE,
        "may_rank_rungs_by_accuracy": ACCURACY_RESOLVABLE,
    },
    "framing": {
        "bare_rung_already_reasons": BARE_ALREADY_REASONS,
        "bare_working_share": bare_working_share,
        "verb": FRAMING_VERB,
        "figure1_left_rung": FIGURE1_LEFT,
    },
    "figure_1": {
        "problem_id": mid_spec["problem_id"],
        "level": mid_spec["level"],
        "question": mid_spec["question"],
        "ground_truth": mid_spec["ground_truth"],
        "prompt_stem": PROMPT_STEM,
        "panels": PANELS,
        "n_candidate_triples": len(triples),
        "specimen_is_fallback": SPECIMEN_IS_FALLBACK,
    },
    "figure_2": {
        "primary_axis": "chain_share",
        "secondary_axis": "accuracy",
        "min_chain_steps": MIN_CHAIN_STEPS,
        "rungs": summary,
        "ranking_worst_to_best": ordered,
        "ranked_by": "chain_share",
        "matches_naive_expectation": ordered == expected,
        "one_shot_few_shot_within_2pts": collapse,
        "chain_resolvable": CHAIN_RESOLVABLE,
        "accuracy_resolvable": ACCURACY_RESOLVABLE,
    },
    "close": close,
    "exemplars": exemplars,
    "few_shot_prompt_example": {"problem_id": mid_spec["problem_id"],
                                "prompt": few_shot_prompt},
}

DERIVED_PATH.write_text(json.dumps(derived, indent=2, ensure_ascii=False), encoding="utf-8")
MANIFEST_PATH.write_text(json.dumps(manifest, indent=2, ensure_ascii=False), encoding="utf-8")

print(f"derived.json   {harness.kb(DERIVED_PATH):>9.1f} KB   <- bundled with the page")
print(f"manifest.json  {harness.kb(MANIFEST_PATH):>9.1f} KB   <- bundled with the page")
print(f"traces.jsonl   {harness.kb(TRACES_PATH):>9.1f} KB   <- published as a download")

gate = "TRANSLATABLE" if (CHAIN_RESOLVABLE and not SPECIMEN_IS_FALLBACK) else "NOT TRANSLATABLE"
print(f"\n{gate}: chain axis {'resolved' if CHAIN_RESOLVABLE else 'DID NOT RESOLVE'}, "
      f"specimen {'real' if not SPECIMEN_IS_FALLBACK else 'FALLBACK'}, "
      f"accuracy {'rankable' if ACCURACY_RESOLVABLE else 'not rankable'}.")

## Notes for re-derivation

- Greedy decoding diverges on a handful of items across hardware (`get_device` enables TF32 on
  CUDA): expect the shape and ranking to reproduce, not every digit.
- The chain axis is a large effect and survives small runs. Accuracy is a small one: at *n* = 24
  it is usually unresolvable and at 160 adjacent rungs still may be. The gate decides, not the eye.
- "A chain" is `MIN_CHAIN_STEPS` steps under this notebook's rule, not a model's judgement. It
  is the same rule figure 1 animates, so disagreeing with it means disagreeing with both.
- The chain axis measures whether reasoning was *written*, not whether it is sound — page 9.
- Changing model, band or cap changes `RUN_ID`; old traces stay as a record and are not replayed.

What the page may claim is [`docs/page_5_refactor.md`](../docs/page_5_refactor.md) §6, checked at
translation ([`docs/page_6_refactor.md`](../docs/page_6_refactor.md) §6).